# 00.- Cargar fichero

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from datetime import datetime
import os


# 11.- Probabilidad variación

In [2]:
def years_ha_subido(df_train:pd.DataFrame, md_inicio:str, md_fin:str)->tuple:
    """Calcula el rendimiento porcentual entre dos fechas del calendario (md_inicio y md_fin)
    para cada año del DataFrame de entrenamiento, y determina cuántos años fueron
    positivos y cuántos negativos en ese intervalo. Devuelve además un DataFrame con
    el beneficio obtenido por cada año.

    Para cada año:
        - Se obtiene el rendimiento acumulado en md_inicio y md_fin.
        - Se calcula el rendimiento del periodo mediante:
              rendimiento = ((1 + rend_fin) / (1 + rend_inicio) - 1) * 100
        - Se clasifica el año como positivo o negativo según el signo del rendimiento.

    Args:
        df_train (pd.DataFrame): DataFrame pivotado donde:
            - El índice son días del año en formato 'MM-DD'.
            - Cada columna representa un año.
            - La última columna suele ser 'rendimiento_medio' y se excluye del cálculo.
        md_inicio (str): Día inicial del intervalo en formato 'MM-DD'.
        md_fin (str): Día final del intervalo en formato 'MM-DD'.

    Returns:
        tuple: df_beneficio_year (pd.DataFrame): DataFrame con columnas ['year', 'beneficio'],
                donde 'beneficio' es el rendimiento porcentual entre md_inicio y md_fin.
                anno_negativo (int): Número de años con rendimiento negativo.
                anno_positivo (int): Número de años con rendimiento positivo.
    """
    
    # Contar rendimiento de cada año
    anno_positivo = 0
    anno_negativo =0
    df_beneficio_year = pd.DataFrame(columns=["year", "beneficio"])

    for y in df_train.columns[:-1]:
        rend_acum_inicio = df_train.loc[md_inicio, y]
        rend_acum_fin = df_train.loc[md_fin, y]


        rend_inicio_a_fin = ((((1 + rend_acum_fin) / (1 + rend_acum_inicio)) -1) * 100).round(2)

        if pd.notna(rend_inicio_a_fin): 
            df_beneficio_year.loc[len(df_beneficio_year)] = [y, rend_inicio_a_fin]
            if rend_inicio_a_fin >= 0: 
                anno_positivo += 1 
            else: 
                anno_negativo += 1
    return df_beneficio_year, anno_negativo, anno_positivo

# 13.- Calculo volatilidad y sharpe ratio

In [3]:
def calculo_volatilidad_sharpe(df_train:pd.DataFrame, md_inicio:str, md_fin:str, rend_inicio_a_fin:float)->tuple:
    """Calcula la volatilidad, el ratio Sharpe y el ratio Sortino para un intervalo
    estacional definido por dos días del año (md_inicio y md_fin), utilizando los
    rendimientos acumulados históricos de cada año.

    Para cada año del DataFrame:
        - Se obtiene el rendimiento acumulado en md_inicio y md_fin.
        - Se calcula el rendimiento del periodo mediante:
              r_periodo = (1 + r_fin) / (1 + r_ini) - 1
        - Se almacena el rendimiento anual del intervalo.

    A partir de todos los rendimientos anuales:
        - La volatilidad se calcula como la desviación típica muestral.
        - El Sharpe ratio se obtiene dividiendo el rendimiento total del periodo
          (rend_inicio_a_fin, expresado en %) entre la volatilidad.
        - El Sortino ratio se calcula igual que el Sharpe, pero usando únicamente
          la desviación típica de los rendimientos negativos. Si no existen
          rendimientos negativos, el Sortino se considera infinito.

    Args:
        df_train (pd.DataFrame): DataFrame pivotado donde:
            - El índice son días del año en formato 'MM-DD'.
            - Cada columna representa un año.
            - La columna 'rendimiento_medio' se excluye del cálculo.
        md_inicio (str): Día inicial del intervalo en formato 'MM-DD'.
        md_fin (str): Día final del intervalo en formato 'MM-DD'.
        rend_inicio_a_fin (float): Rendimiento total del periodo expresado en porcentaje.

    Returns:
        tuple: volatilidad (float): Desviación típica de los rendimientos del periodo.
                sharpe (float): Ratio Sharpe del intervalo.
                sortino (float): Ratio Sortino del intervalo.
    """
    
    # Calculo desviación tipica que es igual que la volatilidad entre md_inicio y md_fin
    cols_anios = [c for c in df_train.columns if c != "rendimiento_medio"]

    rendimientos_periodo = []

    for año in cols_anios:
        r_ini = df_train.loc[md_inicio, año]
        r_fin = df_train.loc[md_fin, año]

        # Fórmula correcta para rendimientos del periodo a partir de los acumulados de inicio y fin
        r_periodo = (1 + r_fin) / (1 + r_ini) - 1

        rendimientos_periodo.append(r_periodo)

    # Calculo desviación típica
    variacion_tipica = np.std(rendimientos_periodo, ddof=1).round(4)
    volatilidad = variacion_tipica

    # Sharpe ratio es una medida del retorno ajustado al riesgo. Valores mayores a 1 son generalmente considerados buenos.
    sharpe = ((rend_inicio_a_fin /100) / volatilidad).round(4)

    # === Sortino ratio === 
    # # Filtrar solo rendimientos negativos 
    downside = [r for r in rendimientos_periodo if r < 0] 
    
    if len(downside) == 0: 
        # Si nunca ha caído, el riesgo negativo es cero → Sortino infinito 
        sortino = np.inf 
    else: 
        downside_std = np.std(downside, ddof=1) 
        sortino = ((rend_inicio_a_fin / 100) / downside_std).round(4)

    
    return volatilidad, sharpe, sortino

# 14.- Primer día habil

In [4]:
# FUNCION para saber el primer dia habil dada una fecha (si es NaN sera el siguiente no NaN)

def primer_dia_habil(df_validacion, md, year):
    """Devuelve el primer día hábil (no NaN) a partir de una fecha dada dentro del
    DataFrame de validación. Si la fecha inicial contiene un valor no disponible
    (NaN) para el año indicado, la función avanza día a día hasta encontrar el
    primer valor válido. Si no existe ningún día hábil posterior, devuelve None.

    El índice del DataFrame debe estar compuesto por días del año en formato 'MM-DD'.

    Args:
        df_validacion (_type_): DataFrame donde:
            - El índice son días del año en formato 'MM-DD'.
            - Cada columna representa un año.
        md (_type_): Fecha inicial en formato 'MM-DD' desde la que se comienza a buscar.
        year (_type_): Año cuya columna se utiliza para comprobar si el día es hábil.

    Returns:
        _type_: El primer día hábil encontrado en formato 'MM-DD'.
                    Devuelve None si no existe ningún día válido desde la fecha dada.
    """
    
    dias = df_validacion.index.tolist()
    pos = dias.index(md)

    for i in range(pos, len(dias)):
        valor = df_validacion.loc[dias[i], year]
        if pd.notna(valor):
            return dias[i]
    return None


# 20.- importar ficheros tratados

In [5]:
# importar los ficheros con los datos ya tratados

def importar_ficheros(input_folder:str, name:str, ticker:str):
    """Carga los ficheros Parquet previamente tratados correspondientes a un activo
    concreto (df_train y df_validacion), extrae la lista de años disponibles y
    genera un DataFrame adicional con un contador de día del año para facilitar
    búsquedas de pautas estacionales.

    El contenido cargado es:
        - df_train: datos históricos de entrenamiento (por ejemplo, 2000–2023),
          con rendimientos acumulados por día del año.
        - df_validacion: datos recientes para validar pautas (por ejemplo, 2024+).
        - anios: lista de columnas de df_validacion, que representan los años disponibles.
        - df_train_busca_pautas: copia de df_train ordenada por fecha e incluyendo
          una columna 'n_dia' que numera los días del año de 1 a 366.

    Args:
        input_folder (str): Carpeta donde se encuentran los ficheros Parquet tratados.
        name (str): Nombre de la empresa (por ejemplo, "Amazon").
        ticker (str): Ticker bursátil de la empresa (por ejemplo, "AMZN").

    Returns:
        _type_:  df_train (pd.DataFrame): Datos históricos de entrenamiento.
                df_validacion (pd.DataFrame): Datos recientes para validación.
                anios (list): Lista de años disponibles en df_validacion.
                df_train_busca_pautas (pd.DataFrame): Versión extendida de df_train con
                    columna 'n_dia' para facilitar búsquedas de patrones.
    """
    
    df_train = pd.read_parquet(f"{input_folder}/{name}_{ticker}_train.parquet")
    df_validacion = pd.read_parquet(f"{input_folder}/{name}_{ticker}_validacion.parquet")

    anios = list(df_validacion.columns)

    # Añado la columna nueva con el numero de dia en el año (de 1 a 366)
    df_train_busca_pautas = df_train.copy()
    df_train_busca_pautas = df_train_busca_pautas.sort_index()
    df_train_busca_pautas["n_dia"] = range(1, len(df_train_busca_pautas) + 1)
    
    return df_train, df_validacion, anios, df_train_busca_pautas


# 21.- Buscar slots de pautas

In [ ]:
def busca_pautas(df_train_busca_pautas:pd.DataFrame, 
                df_train:pd.DataFrame, 
                dia_inicio:int, 
                min_dias:int, 
                max_dias:int, 
                min_probabilidad:float, 
                min_rendimiento:float,
                ultimo_dia_compra:int)->pd.DataFrame:
    """Busca las mejores pautas estacionales del año evaluando todos los intervalos
    posibles entre un día inicial y un rango de duraciones. Para cada día del año
    se prueban múltiples duraciones, se calculan métricas de rendimiento y riesgo,
    y se selecciona la mejor pauta que cumpla unos mínimos exigidos.

    El proceso incluye:
        1. Para cada día del año desde `dia_inicio` hasta `ultimo_dia_compra`:
            - Se identifica la fecha md_inicio correspondiente.
            - Se prueban todas las duraciones entre `min_dias` y `max_dias`.
            - Para cada intervalo (md_inicio → md_fin):
                * Se calcula el rendimiento medio del periodo.
                * Se calcula la duración real en días.
                * Se obtiene el rendimiento del periodo para cada año.
                * Se calcula la probabilidad histórica de obtener un rendimiento positivo.
                * Se calculan volatilidad, Sharpe y Sortino.
                * Se almacena el resultado.

        2. Para cada día inicial, se ordenan los resultados por prioridad:
            - Mayor probabilidad de subida.
            - Mayor Sharpe.
            - Mayor Sortino.
            - Mayor rendimiento.
            - Menor duración.

        3. Se filtran solo las pautas que cumplen:
            - Rendimiento > min_rendimiento.
            - Sharpe > 0.
            - Sortino > 0.
            - Probabilidad > min_probabilidad.

        4. Se guarda únicamente la mejor pauta de cada día inicial.

    Args:
        df_train_busca_pautas (pd.DataFrame): DataFrame con índice 'MM-DD' y columna
            'n_dia' (1–366), usado para mapear días del año a fechas.
        df_train (pd.DataFrame): DataFrame con rendimientos acumulados por día del año
            y por año, incluyendo la columna 'rendimiento_medio'.
        dia_inicio (int): Día del año (1–366) desde el que se empieza a buscar pautas.
        min_dias (int): Duración mínima del intervalo a evaluar.
        max_dias (int): Duración máxima del intervalo a evaluar.
        min_probabilidad (float): Probabilidad mínima de años positivos requerida.
        min_rendimiento (float): Rendimiento mínimo (%) exigido para aceptar una pauta.
        ultimo_dia_compra (int): Último día del año permitido como inicio de pauta.

    Returns:
        pd.DataFrame: Tabla con las mejores pautas encontradas. Contiene:
            - 'md_inicio': fecha inicial en formato 'MM-DD'.
            - 'md_fin': fecha final en formato 'MM-DD'.
            - 'probabilidad_positivo': probabilidad histórica de subida.
            - 'sharpe': Sharpe ratio del intervalo.
            - 'sortino': Sortino ratio del intervalo.
            - 'rend_inicio_a_fin': rendimiento total del periodo (%).
            - 'duracion': duración en días del intervalo.
    """

    # Defino el df donde guardare los resultados de las mejores pautas 
    df_resultados = pd.DataFrame({
        "md_inicio": pd.Series(dtype="object"),
        "md_fin": pd.Series(dtype="object"),
        "probabilidad_positivo": pd.Series(dtype="float"),
        "sharpe": pd.Series(dtype="float"),
        "sortino": pd.Series(dtype="float"),
        "rend_inicio_a_fin": pd.Series(dtype="float"),
        "duracion": pd.Series(dtype="int")
    })

    # ****************************************************************************************************************************
    #        CALCULAR LAS MEJORES TEMPORALIDADES DEL AÑO
    # 1.- Calcular ratios empezando cada dia del año y probando cada duración hasta la maxima
    # 2.- Ordenal los resultados por prioridades y elegir solo la mejor la mejor
    # 3.- Guardar los resultados en el df_resultados
    # ****************************************************************************************************************************

    for dia_i in range(dia_inicio, ultimo_dia_compra):
        md_inicio = df_train_busca_pautas.index[df_train_busca_pautas["n_dia"] == dia_i].tolist()[0]
            
        df_resultados_temp = pd.DataFrame(columns=["md_inicio", "md_fin", "probabilidad_positivo", "sharpe", "sortino", "rend_inicio_a_fin", "duracion"])
        filas = []
        for dia_f in range(dia_i+min_dias, dia_i+max_dias+1):
            md_fin = df_train_busca_pautas.index[df_train_busca_pautas["n_dia"] == dia_f].tolist()[0]
            
            rend_acum_inicio = df_train.loc[md_inicio, "rendimiento_medio"]
            rend_acum_fin = df_train.loc[md_fin, "rendimiento_medio"]
            
            # OJO rendimiento en porcentaje (%)
            rend_inicio_a_fin = ((((1 + rend_acum_fin) / (1 + rend_acum_inicio)) -1) * 100).round(2)
            
            # calculo diferencia dias entre fechas
            diferencia = (datetime.strptime("2000-" + md_fin, "%Y-%m-%d") - datetime.strptime("2000-" + md_inicio, "%Y-%m-%d")).days
            
            df_beneficio_year, anno_negativo, anno_positivo = years_ha_subido(df_train_busca_pautas, md_inicio, md_fin)
            probabilidad_positivo = anno_positivo / (anno_positivo + anno_negativo)
            
            volatilidad, sharpe, sortino = calculo_volatilidad_sharpe(df_train_busca_pautas, md_inicio, md_fin, rend_inicio_a_fin)
            
            # Guardar cada iteración
            filas.append({"md_inicio": md_inicio, 
                        "md_fin": md_fin, 
                        "probabilidad_positivo": probabilidad_positivo, 
                        "sharpe": sharpe, 
                        "sortino": sortino, 
                        "rend_inicio_a_fin": rend_inicio_a_fin, 
                        "duracion": diferencia
                        })
        df_resultados_temp = pd.DataFrame(filas)
        
        # Ordeno por prioridades de los ratios
        df_resultados_temp = df_resultados_temp.sort_values(by= ["probabilidad_positivo", "sharpe", "sortino", "rend_inicio_a_fin", "duracion"],
                                                            ascending=[False, False, False, False, True])
        
        # Me quedo con los que cumplen los minimos por cada ratio
        df_resultados_temp = df_resultados_temp[(
            (df_resultados_temp["rend_inicio_a_fin"] > min_rendimiento) & 
            (df_resultados_temp["sharpe"] > 0) & 
            (df_resultados_temp["sortino"] > 0) &
            (df_resultados_temp["probabilidad_positivo"] > min_probabilidad)
            ) ]
        
        # display(df_resultados_temp)
        if not df_resultados_temp.empty: 
            df_resultados = pd.concat([df_resultados, df_resultados_temp.iloc[[0]]], ignore_index=True)


    return df_resultados


# 22.- Resultado Backtest

🟩 Opción mejor práctica es: 
 - Si valor de día de inicio es festivo(NaN) este año no se valida (Evitando operaciones iguales)
 - Si el valor de día de fin es festivo(NaN) desplazar la salida al siguiente día hábil
 - Esta es la práctica estándar en:
    - trading algorítmico,
    - backtesting profesional,
    - estudios estacionales,
    - papers académicos.

In [ ]:
def resultado_backtest(df_resultados:pd.DataFrame, df_validacion:pd.DataFrame, anios:list)->pd.DataFrame:
    """Ejecuta un backtest de las pautas estacionales seleccionadas utilizando los datos
    de validación. Para cada pauta y para cada año disponible, evalúa si la pauta
    puede aplicarse (día de inicio hábil) y calcula el rendimiento real obtenido.
    Finalmente, genera métricas agregadas que permiten evaluar la robustez de cada pauta.

    El proceso incluye:

    1. Inicialización:
        - Se copia el DataFrame de pautas seleccionadas.
        - Para cada año de validación se crean columnas:
            * test_{year}: indica si la pauta pudo validarse ese año ("SI" o "NO").
            * rend_{year}: rendimiento obtenido en ese año (NaN si no se valida).

    2. Validación año a año:
        - Para cada pauta:
            * Se comprueba si la fecha de inicio es hábil en ese año.
            * Si no lo es, la pauta no se valida ese año.
            * Si lo es, se busca el primer día hábil para la fecha final.
            * Se calcula el rendimiento real del periodo:
                  rendimiento = ((1 + fin) / (1 + ini) - 1) * 100
            * Se almacena el resultado.

    3. Cálculo de métricas agregadas:
        - media_validacion: media de los rendimientos obtenidos en los años válidos.
        - anios_positivos: número de años con rendimiento positivo.
        - pct_acierto: porcentaje de años positivos respecto al total de años de validación.
        - std_validacion: desviación estándar de los rendimientos (riesgo en validación).
        - ranking: ordenación por media_validacion (mayor rendimiento → mejor ranking).

    4. Limpieza y ordenación:
        - Conversión de columnas de rendimiento a valores numéricos.
        - Ordenación final por ranking ascendente (mejores pautas primero).

    Args:
        df_resultados (pd.DataFrame): Pautas seleccionadas en la fase de búsqueda,
            con columnas como 'md_inicio', 'md_fin', 'probabilidad_positivo',
            'sharpe', 'sortino', 'rend_inicio_a_fin', 'duracion'.
        df_validacion (pd.DataFrame): DataFrame con rendimientos acumulados por día
            y por año para validar las pautas.
        anios (list): Lista de años disponibles en df_validacion para realizar el backtest.

    Returns:
        pd.DataFrame: DataFrame final con las pautas evaluadas y las métricas de backtest:
            - test_{year}: si la pauta pudo validarse ese año.
            - rend_{year}: rendimiento obtenido ese año.
            - media_validacion: rendimiento medio en validación.
            - anios_positivos: número de años positivos.
            - pct_acierto: porcentaje de acierto.
            - std_validacion: volatilidad en validación.
            - ranking: ordenación final de las pautas.
    """
    
    df_resultado_backtest = df_resultados.copy()

    contador_si_test = 0
    contador_no_test = 0

    for year in anios:
        
        # Crear columnas vacías para este año 
        df_resultado_backtest[f"test_{year}"] = "" 
        df_resultado_backtest[f"rend_{year}"] = None
        
        for idx, fila in df_resultado_backtest.iterrows():
            ini = fila["md_inicio"]
            
            # 1) Si el inicio es festivo → no validar este año 
            if pd.isna(df_validacion.loc[ini, year]): 
                contador_no_test += 1
                df_resultado_backtest.loc[idx, f"test_{year}"] = "NO" 
                df_resultado_backtest.loc[idx, f"rend_{year}"] = np.nan  # Antes ponia None
                continue
            else:
                fin = primer_dia_habil(df_validacion, fila["md_fin"], year)
                df_resultado_backtest.loc[idx, f"test_{year}"] = "SI"
                contador_si_test += 1

                valor_ini = df_validacion.loc[ini,year]
                valor_fin = df_validacion.loc[fin,year]

                rend_test_ini_a_fin = ((((1 + valor_fin) / (1 + valor_ini)) -1) * 100).round(2)
                
                df_resultado_backtest.loc[idx, f"rend_{year}"] = rend_test_ini_a_fin
                
                
        
    df_final = df_resultado_backtest.copy()

    # Lista de columnas de rendimiento
    cols_rend = [f"rend_{y}" for y in anios]

    # Media de validación de cada ventana a lo largo de los años de validacion. Para saber qué ventanas son más rentables en promedio.
    df_final["media_validacion"] = df_final[cols_rend].mean(axis=1)

    # Número de años con rendimientos positivos en la ventana. Para medir la consistencia de la ventana.
    df_final["anios_positivos"] = (df_final[cols_rend] > 0).sum(axis=1)

    # Porcentaje de acierto. Convierte el número de años positivos en un porcentaje para ver qué ventanas funcionan la mayoría de los años
    df_final["pct_acierto"] = (df_final["anios_positivos"] / len(anios) * 100).round(1)

    # Desviación estándar (volatilidad de la validación). Para medir el riesgo de la ventana. Cuanto más baja, más estable y predecible
    df_final["std_validacion"] = df_final[cols_rend].std(axis=1).round(2)

    # Ranking final (mayor media y mayor robustez). Para identificar rápidamente las mejores ventanas estacionales
    df_final["ranking"] = df_final["media_validacion"].rank(ascending=False)



    # Convertir todos los valores a numeros y si no puede convertirlo lo pondra como NaN (errors="coerce")
    df_final[cols_rend] = df_final[cols_rend].apply(pd.to_numeric, errors="coerce")

    # Ordeno el df por la columna ranking
    df_final = df_final.sort_values("ranking")
    
        
        
        
    return df_final

# 23.- Resumen backtest

In [8]:
def resumen_backtest(df_final:pd.DataFrame, anios:list, capital_por_ventana:float, capital_inicial_anual:float)->pd.DataFrame:
    """Genera un resumen económico completo del backtest a partir de los resultados
    obtenidos por ventana estacional. Calcula beneficios por ventana y por año,
    métricas agregadas anuales y un resumen global del periodo evaluado.

    El proceso incluye:

    1. Cálculo del beneficio por ventana:
        - Para cada año de validación, se multiplica el rendimiento de cada ventana
          por el capital asignado a esa ventana (`capital_por_ventana`).

    2. Beneficio total anual:
        - Se suman los beneficios de todas las ventanas válidas de cada año.
        - Se calcula el capital final del año:
              capital_final = capital_inicial_anual + beneficio_total

    3. Métricas anuales adicionales:
        - ventanas_operadas: número de ventanas válidas en cada año.
        - drawdown_max: peor operación del año (mínimo beneficio).
        - mejor_operacion: mejor operación del año (máximo beneficio).
        - retorno_%: rendimiento anual sobre el capital inicial.

    4. Construcción del resumen anual:
        - Se genera un DataFrame con:
            * capital_inicial
            * capital_final
            * beneficio_total
            * retorno_%
            * ventanas_operadas
            * drawdown_max
            * mejor_operacion

    5. Resumen global del periodo completo:
        - Suma de beneficios totales.
        - Capital final acumulado.
        - Total de ventanas operadas.
        - Peor drawdown del periodo.
        - Mejor operación del periodo.
        - Retorno total porcentual.

    Args:
        df_final (pd.DataFrame): Resultados del backtest por ventana, incluyendo
            columnas `rend_{year}` con los rendimientos por año.
        anios (list): Lista de años utilizados en la validación.
        capital_por_ventana (float): Capital asignado a cada ventana operada.
        capital_inicial_anual (float): Capital inicial disponible cada año.

    Returns:
        pd.DataFrame: Resumen anual y global del backtest, con métricas económicas
        clave para evaluar la rentabilidad y estabilidad de las pautas estacionales.
    """
    
    # Beneficio por ventana y año
    for year in anios:
        df_final[f"benef_{year}"] = (df_final[f"rend_{year}"] / 100) * capital_por_ventana

    # Beneficio total del año (sumando todas las ventanas válidas)
    for year in anios:
        df_final[f"benef_total_{year}"] = df_final[f"benef_{year}"].sum()

    # Capital final del año
    for year in anios:
        df_final[f"capital_final_{year}"] = capital_inicial_anual + df_final[f"benef_total_{year}"]


    # ************** Beneficio total del año  ************

    beneficios_totales = {
        year: df_final[f"benef_{year}"].sum()
        for year in anios
    }


    # Capital final del año

    capital_final = {
        year: capital_inicial_anual + beneficios_totales[year]
        for year in anios
    }

    # Número de ventanas operadas por año

    ventanas_operadas = {
        year: df_final[f"rend_{year}"].notna().sum()
        for year in anios
    }

    # Drawdown anual (pérdida máxima en euros)

    drawdown_anual = {
        year: df_final[f"benef_{year}"].min()
        for year in anios
    }

    # la mejor operación del año
    mejor_operacion_anual = {
        year: df_final[f"benef_{year}"].max()
        for year in anios
    }

    # Retorno anual sobre el capital

    retorno_anual = {
        year: (beneficios_totales[year] / capital_inicial_anual) * 100
        for year in anios
    }


    # **************** Construcción del resumen anual completo **********************

    resumen_anual = pd.DataFrame({
        "capital_inicial": capital_inicial_anual,    
        "capital_final": capital_final,
        "beneficio_total": beneficios_totales,
        "retorno_%": retorno_anual,
        "ventanas_operadas": ventanas_operadas,
        "drawdown_max": drawdown_anual,
        "mejor_operacion": mejor_operacion_anual
    }).T


    # Añado el completo de todo el periodo
    resumen_anual["completo"] = np.nan
    resumen_anual.loc["capital_inicial","completo"] = capital_inicial_anual
    resumen_anual.loc["beneficio_total","completo"] = resumen_anual.loc["beneficio_total"].sum()
    resumen_anual.loc["capital_final","completo"] = capital_inicial_anual + resumen_anual.loc["beneficio_total","completo"]
    resumen_anual.loc["ventanas_operadas","completo"] = resumen_anual.loc["ventanas_operadas"].sum()
    resumen_anual.loc["drawdown_max","completo"] = resumen_anual.loc["drawdown_max"].min()
    resumen_anual.loc["mejor_operacion","completo"] = resumen_anual.loc["mejor_operacion"].max()
    resumen_anual.loc["retorno_%","completo"] = (resumen_anual.loc["capital_final","completo"] / capital_inicial_anual - 1) * 100
    resumen_anual = resumen_anual.round(2)
    # print(f"RESUMEN BACKTEST {name}")
    
    return resumen_anual

# 900.- Principal

Difinir condiciones de operación:
 - 1.- Comprar entre el 1 de enero (01-01) y el 5 de Diciembre (12-05)
 - 2.- Duración entre 2 y 20 días
 - 3.- Intentar no comprar en festivo

In [ ]:
"""
Ejecuta el proceso completo de búsqueda, validación y backtest de pautas estacionales
para un conjunto de acciones del SP500 Top 25. El script aplica filtros definidos por
el usuario, evalúa miles de combinaciones de fechas, selecciona las mejores pautas,
realiza un backtest sobre años recientes y genera un resumen consolidado.

El flujo general incluye:

1. Definición de parámetros:
    - Día inicial y final permitido para iniciar una pauta.
    - Duración mínima y máxima de cada ventana estacional.
    - Probabilidad mínima de subida y rendimiento mínimo exigido.
    - Capital asignado por ventana y capital inicial anual para el backtest.

2. Iteración por cada acción del SP500 Top 25 seleccionada:
    - Carga de los ficheros tratados (train y validación).
    - Construcción del DataFrame auxiliar con numeración de días del año.
    - Búsqueda de pautas estacionales mediante:
        * Evaluación de todas las combinaciones inicio–fin dentro de los límites.
        * Cálculo de probabilidad histórica, Sharpe, Sortino y rendimiento.
        * Filtrado por criterios mínimos.
        * Selección de la mejor pauta para cada día del año.
    - Ejecución del backtest:
        * Validación año a año según días hábiles.
        * Cálculo de rendimientos reales por ventana.
        * Métricas agregadas: media, acierto, volatilidad, ranking.
    - Generación del resumen económico anual:
        * Beneficio total por año.
        * Capital final.
        * Número de ventanas operadas.
        * Drawdown máximo y mejor operación.
        * Resumen global del periodo completo.
    - Guardado de resultados individuales en formato Parquet.

3. Consolidación final:
    - Unión de los resúmenes anuales de todas las acciones.
    - Ordenación por acción y periodo.
    - Exportación del DataFrame consolidado a Parquet.
    - Visualización del resultado final.

Este script permite automatizar el análisis estacional completo para múltiples activos,
incluyendo búsqueda de pautas, validación histórica y evaluación económica del backtest.
"""


# ****************** PARAMETROS DE FILTRO ***********************************
dia_inicio =1
ultimo_dia_compra =341
min_dias = 2
max_dias = 20
min_probabilidad = 0.65
min_rendimiento = 1   # en porcentaje. Si es 2 es 2% y si es 0.2 es 0,2%
capital_por_ventana = 1000
capital_inicial_anual = 10000
df_total_backtest_ordenado = []

# Entrada de acciones. EN FUTURO HABRA UN BUCLE DE LA LISTA TOTAL
input_folder = "SP500_Top25_tratados"


sp500_top25 = {
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "Amazon": "AMZN",
    "Meta": "META",
    "Alphabet Class A": "GOOGL",
    "Alphabet Class C": "GOOG",
    "Berkshire Hathaway": "BRK-B",
    "Eli Lilly": "LLY",
    "Broadcom": "AVGO",
    "Tesla": "TSLA",
    "JPMorgan": "JPM",
    "UnitedHealth": "UNH",
    "Visa": "V",
    "Exxon Mobil": "XOM",
    "Mastercard": "MA",
    "Johnson & Johnson": "JNJ",
    "Procter & Gamble": "PG",
    "Home Depot": "HD",
    "Costco": "COST",
    "Merck": "MRK",
    "Chevron": "CVX",
    "AbbVie": "ABBV",
    "PepsiCo": "PEP",
    "Walmart": "WMT"
}

for name, ticker in sp500_top25.items():

    df_train, df_validacion, anios, df_train_busca_pautas = importar_ficheros(input_folder, name, ticker)

    df_resultados = busca_pautas(df_train_busca_pautas, 
                    df_train, 
                    dia_inicio, 
                    min_dias, 
                    max_dias, 
                    min_probabilidad, 
                    min_rendimiento,
                    ultimo_dia_compra)

    df_final = resultado_backtest(df_resultados, df_validacion, anios)

    # Guardar df_final
    output_folder_out = "SP500_Top25_resultados"

    try:
        os.makedirs(output_folder_out, exist_ok=True)

        df_final.to_parquet(f"{output_folder_out}/{name}_{ticker}_final.parquet", index=True)

    except Exception as e: 
        print(f"❌ Error inesperado guardando: {e}")


    df_resumen_anual = resumen_backtest(df_final, anios, capital_por_ventana, capital_inicial_anual)

    # Convertir todas las filas en un DataFrame normal
    df_temp = df_resumen_anual.T.copy()

    # Añadir columnas con el nombre de la acción y el periodo (índice)
    df_temp["name"] = name
    df_temp["periodo"] = df_temp.index

    # Guardar en la lista
    df_total_backtest_ordenado.append(df_temp)

# FUERA DEL BUCLE ******************************************************************************

# Unir todo en un único DataFrame
df_total_backtest_ordenado = pd.concat(df_total_backtest_ordenado, ignore_index=True)

# Ordenar por acción y periodo
df_total_backtest_ordenado = df_total_backtest_ordenado.sort_values(["name", "periodo"])

try:
    df_total_backtest_ordenado["periodo"] = df_total_backtest_ordenado["periodo"].astype(str)
    df_total_backtest_ordenado.to_parquet(f"{output_folder_out}/total_backtest_ordenado.parquet", index=True)

except Exception as e: 
    print(f"❌ Error inesperado guardando: {e}")



print(f"TOTAL BACKTEST ORDENADO")
display(df_total_backtest_ordenado)


c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Ignacio\anaconda3\envs\mt5pyt

TOTAL BACKTEST ORDENADO


,capital_inicial,capital_final,beneficio_total,retorno_%,ventanas_operadas,drawdown_max,mejor_operacion,name,periodo
66,10000.0,9564.3,-435.7,-4.36,170.0,-175.1,102.1,AbbVie,2024
67,10000.0,10940.2,940.2,9.40,177.0,-156.0,127.4,AbbVie,2025
68,10000.0,10504.5,504.5,5.05,347.0,-175.1,127.4,AbbVie,completo
15,10000.0,11845.9,1845.9,18.46,165.0,-102.5,151.9,Alphabet Class A,2024
16,10000.0,15161.0,5161.0,51.61,179.0,-162.5,194.4,Alphabet Class A,2025
...,...,...,...,...,...,...,...,...,...
40,10000.0,10482.7,482.7,4.83,155.0,-134.9,107.5,Visa,2025
41,10000.0,11826.5,1826.5,18.26,300.0,-134.9,107.5,Visa,completo
72,10000.0,11457.9,1457.9,14.58,92.0,-27.4,87.7,Walmart,2024
73,10000.0,10176.6,176.6,1.77,98.0,-115.1,159.2,Walmart,2025


In [17]:
df_priorizado = df_total_backtest_ordenado[df_total_backtest_ordenado["periodo"] == "completo"]
df_priorizado.sort_values(by=["retorno_%"],ascending=False)

,capital_inicial,capital_final,beneficio_total,retorno_%,ventanas_operadas,drawdown_max,mejor_operacion,name,periodo
8,10000.0,18305.0,8305.0,83.05,263.0,-167.9,299.5,Nvidia,completo
29,10000.0,18293.6,8293.6,82.94,333.0,-237.4,500.3,Broadcom,completo
17,10000.0,17006.9,7006.9,70.07,344.0,-162.5,194.4,Alphabet Class A,completo
20,10000.0,16930.1,6930.1,69.30,342.0,-162.9,193.2,Alphabet Class C,completo
32,10000.0,15936.5,5936.5,59.36,336.0,-220.4,456.1,Tesla,completo
14,10000.0,13157.8,3157.8,31.58,340.0,-191.6,203.8,Meta,completo
35,10000.0,12376.0,2376.0,23.76,196.0,-84.1,151.2,JPMorgan,completo
47,10000.0,12112.2,2112.2,21.12,335.0,-103.3,140.2,Mastercard,completo
41,10000.0,11826.5,1826.5,18.26,300.0,-134.9,107.5,Visa,completo
74,10000.0,11634.5,1634.5,16.35,190.0,-115.1,159.2,Walmart,completo
